# Fine-tune the encoder on a free GPU

This notebook trains DistilBERT on the NHTSA complaints, once for the random split and once for the time split.
It works on Google Colab and on Kaggle.

**Before you start**
1. Turn on a GPU: Colab, Runtime > Change runtime type > T4 GPU. Kaggle, Settings > Accelerator > GPU, and turn Internet on.
2. Run the cells from top to bottom. The whole notebook takes about 40 to 60 minutes (data preparation about 10 minutes, training about 10 to 15 minutes per split).
3. At the end, download `finetune_results.zip` and unzip it into the `results/` folder of your local copy of the repository.

The notebook prepares the data again with the same code and the same seed, and then **checks that the splits are identical**
to the ones in the repository. If the check fails, stop: the results would not be comparable.

In [ ]:
REPO_URL = "https://github.com/Shashank-2902/complaint-classifier"   # change this if your repository has another address
!git clone {REPO_URL} complaint-classifier
%cd complaint-classifier
!pip install -q transformers pandas scikit-learn

In [ ]:
# Keep the report from the repository, to compare with the data that is prepared here.
!cp results/data_report.json /content/committed_report.json
!python -m src.prepare.download
!python -m src.prepare.run

In [ ]:
import json
committed = json.load(open("/content/committed_report.json"))
fresh = json.load(open("results/data_report.json"))
same = all(committed["splits"][s][p]["fingerprint"] == fresh["splits"][s][p]["fingerprint"]
           for s in ("random", "time") for p in ("train", "val", "test"))
print("splits identical to the repository:", same)
assert same, "The prepared splits differ from the repository. Do not continue."

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Quick check that everything works (a few hundred complaints, never touches the test set).
!python -m src.finetune.train --scheme random --smoke

In [ ]:
# The real training runs. Each one opens the test set once, at the end.
!python -m src.finetune.train --scheme random
!python -m src.finetune.train --scheme time

In [ ]:
# Pack the results to download. The access log shows that each test set was opened once.
!grep finetuned_encoder results/test_access_log.jsonl > results/test_access_log_finetuned.jsonl
!zip -r finetune_results.zip results/predictions/finetuned_encoder__*.csv results/predictions/finetuned_encoder__*.meta.json results/test_access_log_finetuned.jsonl
from google.colab import files   # on Kaggle, open the 'Output' tab instead
files.download("finetune_results.zip")